In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.chains.retrieval_qa.base import RetrievalQA
from langchain.document_loaders import CSVLoader
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.indexes.vectorstore import VectorstoreIndexCreator
from langchain.evaluation.qa import QAGenerateChain, QAEvalChain
import langchain

In [2]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:4B",
    # model="qwen3:1.7B",
    temperature=0.9,
    verbose=True,
    extract_reasoning=True,
)
embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")

In [3]:
loader = CSVLoader(file_path="../data/04-OutdoorClothingCatalog_1000.csv")
documents = loader.load()

In [4]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings,
).from_loaders([loader])

/home/eugene/projects/deeplearning.ai/.venv/lib/python3.13/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [5]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.vectorstore.as_retriever(),
    verbose=True,
    chain_type_kwargs={
        "document_separator": "<<<<>>>>>",
    },
)

In [6]:
documents[10:12]

[Document(metadata={'source': '../data/04-OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported."),
 Document(metadata={'source': '../data/04-OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is

### Hard-coded examples

In [15]:
manual_examples = [
    {"query": "Do the Cozy Comfort Pullover Set have side pockets?", "answer": "Yes"},
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection",
    },
]

### LLM-Generated examples

In [14]:
langchain.debug = False

qa_gen_chain = QAGenerateChain.from_llm(llm=llm)
examples = qa_gen_chain.apply_and_parse([{"doc": document} for document in documents[:5]])
examples = [doc["qa_pairs"] for doc in examples]
examples[0]

/home/eugene/projects/deeplearning.ai/.venv/lib/python3.13/site-packages/langchain/chains/llm.py:370: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


{'query': "What is the recommended sizing guideline for the Women's Campside Oxfords?  ",
 'answer': 'The product recommends ordering regular shoe size. For half sizes not available, customers should order up to the next whole size.'}

In [28]:
examples += manual_examples
examples

[{'query': "What is the recommended sizing guideline for the Women's Campside Oxfords?  ",
  'answer': 'The product recommends ordering regular shoe size. For half sizes not available, customers should order up to the next whole size.'},
 {'query': 'What environmental benefits does the Recycled Waterhog Dog Mat provide, and what materials is it made from?  ',
  'answer': 'The Recycled Waterhog Dog Mat helps keep dirt and water off floors while reducing plastic waste by using 94% recycled materials. It prevents plastic from ending up in landfills, trails, and oceans, making it an eco-friendly choice. The mat is constructed from 24 oz. polyester fabric and includes a rubber backing.'},
 {'query': "What is the sun protection rating of the Infant and Toddler Girls' Coastal Chill Swimsuit, and what percentage of harmful rays does it block?  ",
  'answer': "The swimsuit has a UPF 50+ rated fabric, which blocks 98% of the sun's harmful rays."},
 {'query': 'What is the sun protection rating of

In [17]:
langchain.debug = True

qa.invoke(examples[0]["query"])

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "What is the recommended sizing guideline for the Women's Campside Oxfords?  "
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "What is the recommended sizing guideline for the Women's Campside Oxfords?  ",
  "context": ": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage 

{'query': "What is the recommended sizing guideline for the Women's Campside Oxfords?  ",
 'result': "\n\nThe recommended sizing guideline for the Women's Campside Oxfords is to **order regular shoe size**. For half sizes not offered, **order up to the next whole size**."}

In [29]:
langchain.debug = False

predictions = qa.apply(examples)
predictions



> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


[{'query': "What is the recommended sizing guideline for the Women's Campside Oxfords?  ",
  'answer': 'The product recommends ordering regular shoe size. For half sizes not available, customers should order up to the next whole size.',
  'result': "\n\nThe recommended sizing guideline for the Women's Campside Oxfords is to **order regular shoe size**. For half sizes not offered, **order up to the next whole size**."},
 {'query': 'What environmental benefits does the Recycled Waterhog Dog Mat provide, and what materials is it made from?  ',
  'answer': 'The Recycled Waterhog Dog Mat helps keep dirt and water off floors while reducing plastic waste by using 94% recycled materials. It prevents plastic from ending up in landfills, trails, and oceans, making it an eco-friendly choice. The mat is constructed from 24 oz. polyester fabric and includes a rubber backing.',
  'result': "\n\nThe Recycled Waterhog Dog Mat provides environmental benefits by using **24 oz. polyester fabric made from

In [30]:
eval_chain = QAEvalChain.from_llm(llm=llm)

In [31]:
langchain.debug = False

graded_outputs = eval_chain.evaluate(examples=examples, predictions=predictions)
graded_outputs

[{'results': '\n\nGRADE: CORRECT'},
 {'results': '\n\nGRADE: CORRECT'},
 {'results': '\n\nGRADE: CORRECT'},
 {'results': '\n\nGRADE: CORRECT'},
 {'results': '\n\nGRADE: CORRECT'},
 {'results': '\n\nGRADE: CORRECT'},
 {'results': '\n\nGRADE: CORRECT'}]

In [32]:
for i, eg in enumerate(examples):
    print("=============================================")
    print(f"Example {i}:")
    print("Question: " + predictions[i]["query"])
    print("Real Answer: " + predictions[i]["answer"])
    print("Predicted Answer: " + predictions[i]["result"])
    print("Predicted Grade: " + graded_outputs[i]["results"])
    print()

Example 0:
Question: What is the recommended sizing guideline for the Women's Campside Oxfords?  
Real Answer: The product recommends ordering regular shoe size. For half sizes not available, customers should order up to the next whole size.
Predicted Answer: 

The recommended sizing guideline for the Women's Campside Oxfords is to **order regular shoe size**. For half sizes not offered, **order up to the next whole size**.
Predicted Grade: 

GRADE: CORRECT

Example 1:
Question: What environmental benefits does the Recycled Waterhog Dog Mat provide, and what materials is it made from?  
Real Answer: The Recycled Waterhog Dog Mat helps keep dirt and water off floors while reducing plastic waste by using 94% recycled materials. It prevents plastic from ending up in landfills, trails, and oceans, making it an eco-friendly choice. The mat is constructed from 24 oz. polyester fabric and includes a rubber backing.
Predicted Answer: 

The Recycled Waterhog Dog Mat provides environmental benef